In [15]:
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV, FrozenEstimator
from model.metrics import compute_model_output_metrics
import seaborn as sns
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

In [16]:
sns.set_theme(style='darkgrid')
sns.set_context('paper')

In [17]:
X_train, y_train = np.load('train_x.npy'), np.load('train_y.npy')
X_test, y_test = np.load('test_x.npy'), np.load('test_y.npy')

In [18]:
def describe_model(model):
    y_preds = model.predict(X_test)
    probs = model.predict_proba(X_test)
    stats = compute_model_output_metrics(y_test, y_preds, probs)
    print(f'Accuracy: {round(stats['acc'] * 100, 2)}%')
    print(f'Expected Calibration Error: {round(stats['ece'] * 100, 2)}%')
    print(f'Spread: {round(stats['spread'], 4)}')
    print(f'Binary Cross Entropy Loss: {round(stats['bce'], 4)}')
    plt.figure(figsize=(8, 5))
    sns.histplot(x=probs[:, 0], kde=True, bins=20)
    plt.xlabel('Probability')
    plt.title('Estimated Win Probability Frequencies')
    plt.show()

In [19]:
lr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression())
])

In [20]:
lr_model.fit(X_train, y_train)

/home/fahd/.pyenv/versions/3.12.2/envs/lolenv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


Pipeline(steps=[('scaler', StandardScaler()), ('logreg', LogisticRegression())])

In [13]:
describe_model(lr_model)

NameError: name 'describe_model' is not defined

In [14]:
mlp = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(64),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        batch_size=32,
        learning_rate_init=1e-3,
        max_iter=300,
        early_stopping=True,
        random_state=42
    ))
])

In [ ]:
calibrated_mlp = CalibratedClassifierCV(
    estimator=mlp,
    method="isotonic",
    cv=5,
)

In [2]:
calibrated_mlp.fit(X_train, y_train)

NameError: name 'calibrated_mlp' is not defined

In [3]:
describe_model(calibrated_mlp)

NameError: name 'describe_model' is not defined

In [4]:
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    eval_metric="logloss",
    random_state=42
)

NameError: name 'XGBClassifier' is not defined

In [5]:
xgb_calibrated = CalibratedClassifierCV(
    estimator=xgb,
    method="isotonic",
    cv=5
)

NameError: name 'xgb' is not defined

In [6]:
xgb_calibrated.fit(X_train, y_train)

NameError: name 'xgb_calibrated' is not defined

In [ ]:
describe_model(xgb_calibrated)

In [ ]:
import joblib
joblib.dump(calibrated_mlp, 'model.pkl')